# Phase-Amplitude

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert, butter, filtfilt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# Simulation parameters
sfreq = 1000             # Sampling frequency in Hz
duration = 10           # Duration in seconds
n_samples = int(sfreq * duration)
times = np.arange(n_samples) / sfreq

# 1. Generate low-frequency (theta) signal
lf_freq = 4  # Hz
lf_signal = np.sin(2 * np.pi * lf_freq * times) # predictor for low-frequency phase (e.g, speech rhythm)

# Extract phase
lf_phase = np.angle(hilbert(lf_signal))

# 2. Generate high-frequency (gamma) signal
hf_freq = 80  # Hz
hf_carrier = np.sin(2 * np.pi * hf_freq * times)

# --- Modulate HF amplitude by LF phase (PAC) ---
modulation_index = 0.8  # --> amplitude envelope
hf_envelope = 1 + modulation_index * np.cos(lf_phase)  # target/response envelope modulated by LF phase
hf_pac_signal = hf_envelope * hf_carrier # phase-to-amplitude mapping

# --- Simulated EEG = PAC signal + Gaussian noise ---
noise = np.random.randn(n_samples) * 0.5
sim_eeg = hf_pac_signal + noise # simulated EEG trace that contains known PAC

# Time lags
tmin = -0.1   # in seconds (−100 ms)
tmax = 0.4    # in seconds (+400 ms)
lags = np.arange(int(tmin * sfreq), int(tmax * sfreq) + 1)
n_lags = len(lags)

# --- Create time-lagged predictor matrix ---
# Each column = predictor shifted by a time lag
def make_lagged_matrix(predictor, lags):
    padded = np.pad(predictor, (np.max(np.abs(lags)),), mode='constant')
    X_lagged = np.stack([
        padded[np.max(np.abs(lags)) + lag : np.max(np.abs(lags)) + lag + len(predictor)]
        for lag in lags
    ], axis=1)
    return X_lagged

X_lagged = make_lagged_matrix(lf_signal, lags)

# --- Standardize predictors and response ---
X_std = StandardScaler().fit_transform(X_lagged)
Y_std = StandardScaler().fit_transform(sim_eeg.reshape(-1, 1)).ravel()

# --- Fit mTRF using ridge regression ---
ridge = Ridge(alpha=1.0)  # alpha = regularization strength
ridge.fit(X_std, Y_std)

# --- Extract TRF weights ---
trf_weights = ridge.coef_

# --- Plot TRF ---
plt.figure(figsize=(6, 4))
plt.plot(lags * 1000, trf_weights, label='mTRF (PAC)')
plt.axhline(0, color='gray', linestyle='--')
plt.title('Phase-Amplitude Coupling')
plt.xlabel('Lag (ms)')
plt.ylabel('TRF weight (a.u.)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# Setup
sfreq = 1000
duration = 10
n_samples = int(sfreq * duration)
times = np.arange(n_samples) / sfreq

# Low-freq carrier
lf_freq = 4
lf_signal = np.sin(2 * np.pi * lf_freq * times)
lf_amp = np.abs(hilbert(lf_signal))  # low-freq amplitude envelope

# --- Step 1: Generate slow amplitude modulator amp_mod(t) ---
amp_mod = np.zeros(n_samples)
for _ in range(5):  # sum of 5 random sinusoids < 1 Hz
    f_i = np.random.uniform(0.05, 0.5)  # random freq <= 0.5 Hz
    phi_i = np.random.uniform(0, 2 * np.pi)
    amp_mod += np.cos(2 * np.pi * f_i * times + phi_i)
amp_mod = (amp_mod - amp_mod.min()) / (amp_mod.max() - amp_mod.min())  # normalize to [0,1]

# --- Step 2: Generate hf_amp_ind(t) ---
hf_amp_ind = np.zeros(n_samples)
for _ in range(5):  # similar idea: unrelated slow rhythm
    f_i = np.random.uniform(0.05, 0.5)
    phi_i = np.random.uniform(0, 2 * np.pi)
    hf_amp_ind += np.cos(2 * np.pi * f_i * times + phi_i)
hf_amp_ind = (hf_amp_ind - hf_amp_ind.min()) / (hf_amp_ind.max() - hf_amp_ind.min())

# --- Step 3: Final high-frequency envelope ---
hf_envelope = (1 - amp_mod) * hf_amp_ind + amp_mod * lf_amp

# HF carrier (gamma band)
hf_freq = 80
hf_carrier = np.sin(2 * np.pi * hf_freq * times)
hf_signal = hf_envelope * hf_carrier + 0.3 * np.random.randn(n_samples)

# Plot
plt.figure(figsize=(10, 4))
plt.plot(times, hf_signal, label='Simulated PAC EEG')
plt.plot(times, hf_envelope, label='PAC Envelope', alpha=0.7)
plt.plot(times, amp_mod, label='amp_mod(t)', linestyle='--')
plt.title("Data-Inspired PAC Simulation (Weissbart-style)")
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.tight_layout()
plt.show()


# Amplitude- Amplitude & Phase- Phase

In [ ]:
# Re-import after state reset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# Simulate base parameters
sfreq = 1000  # Hz
t = np.arange(0, 10, 1/sfreq)
n_samples = len(t)

# Low-frequency (LF) signal for amplitude–amplitude coupling
lf_signal = np.sin(2 * np.pi * 4 * t)  # 4 Hz
lf_envelope = np.abs(lf_signal)

# High-frequency (HF) carrier signal
hf_signal = np.sin(2 * np.pi * 40 * t)  # 40 Hz

# Amplitude–Amplitude Coupling (AAC)
hf_modulated = hf_signal * (1 + 0.5 * lf_envelope)  # --> hf_envelope signal
sim_eeg_aac = hf_modulated + 0.1 * np.random.randn(n_samples)
# hf_amplitude =
# lf_amplitude =
# fromulation ,
# lf_amplitude =  sepeech x t (0,1)



# Phase–Phase Coupling (PPC): simulate a locked phase relationship
lf_phase = np.angle(np.exp(1j * 2 * np.pi * 4 * t))
hf_phase = np.angle(np.exp(1j * 2 * np.pi * 40 * t))
combined_phase = np.cos(lf_phase - hf_phase)  # Phase locking
sim_eeg_ppc = combined_phase + 0.1 * np.random.randn(n_samples)

# Time lags
tmin, tmax = -0.1, 0.4  # seconds
lags = np.arange(int(tmin * sfreq), int(tmax * sfreq) + 1)
n_lags = len(lags)

def make_lagged_matrix(predictor, lags):
    padded = np.pad(predictor, (np.max(np.abs(lags)),), mode='constant')
    X_lagged = np.stack([
        padded[np.max(np.abs(lags)) + lag : np.max(np.abs(lags)) + lag + len(predictor)]
        for lag in lags
    ], axis=1)
    return X_lagged

# Fit mTRF function
def fit_mtrf(predictor, response):
    X_lagged = make_lagged_matrix(predictor, lags)
    X_std = StandardScaler().fit_transform(X_lagged)
    Y_std = StandardScaler().fit_transform(response.reshape(-1, 1)).ravel()
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_std, Y_std)
    return ridge.coef_

# AAC: amplitude-amplitude
trf_aac = fit_mtrf(lf_envelope, sim_eeg_aac)

# PPC: phase-phase
trf_ppc = fit_mtrf(lf_phase, sim_eeg_ppc)
# phasse - fluctions,
# hf_freq_ind =
# lf_freq_ind =


# f(t)= freq ' (t)
# 40 hz




# Plotting both
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(lags * 1000, trf_aac, label='AAC mTRF', color='darkorange')
plt.title('Amplitude-Amplitude Coupling')
plt.axhline(0, color='gray', linestyle='--')
plt.xlabel('Lag (ms)')
plt.ylabel('TRF weight')

plt.subplot(1, 2, 2)
plt.plot(lags * 1000, trf_ppc, label='PPC mTRF', color='teal')
plt.title('Phase-Phase Coupling')
plt.axhline(0, color='gray', linestyle='--')
plt.xlabel('Lag (ms)')

plt.tight_layout()
plt.show()


# Combined

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# ----------------------------
# Simulation parameters
# ----------------------------
sfreq = 1000  # Sampling frequency (Hz)
duration = 10  # seconds
n_samples = int(sfreq * duration)
times = np.arange(n_samples) / sfreq

# ----------------------------
# Phase-Amplitude Coupling (PAC)
# ----------------------------
lf_freq = 4
hf_freq = 80
lf_signal_pac = np.sin(2 * np.pi * lf_freq * times)
lf_phase_pac = np.angle(hilbert(lf_signal_pac))
hf_carrier = np.sin(2 * np.pi * hf_freq * times)
hf_envelope = 1 + 0.8 * np.cos(lf_phase_pac)
sim_eeg_pac = hf_carrier * hf_envelope + np.random.randn(n_samples) * 0.5

# ----------------------------
# Amplitude-Amplitude Coupling (AAC)
# ----------------------------
lf_signal_aac = np.sin(2 * np.pi * 4 * times)
lf_envelope_aac = np.abs(lf_signal_aac)
hf_signal_aac = np.sin(2 * np.pi * 40 * times)
sim_eeg_aac = hf_signal_aac * (1 + 0.5 * lf_envelope_aac) + 0.1 * np.random.randn(n_samples)

# ----------------------------
# Phase-Phase Coupling (PPC)
# ----------------------------
lf_phase_ppc = np.angle(np.exp(1j * 2 * np.pi * 4 * times))
hf_phase_ppc = np.angle(np.exp(1j * 2 * np.pi * 40 * times))
combined_phase = np.cos(lf_phase_ppc - hf_phase_ppc)
sim_eeg_ppc = combined_phase + 0.1 * np.random.randn(n_samples)

# ----------------------------
# Time lags
# ----------------------------
tmin, tmax = -0.1, 0.4  # seconds
lags = np.arange(int(tmin * sfreq), int(tmax * sfreq) + 1)

def make_lagged_matrix(predictor, lags):
    pad = np.max(np.abs(lags))
    padded = np.pad(predictor, (pad,), mode='constant')
    return np.stack([
        padded[pad + lag : pad + lag + len(predictor)] for lag in lags
    ], axis=1)

def fit_mtrf(predictor, response):
    X_lagged = make_lagged_matrix(predictor, lags)
    X_std = StandardScaler().fit_transform(X_lagged)
    Y_std = StandardScaler().fit_transform(response.reshape(-1, 1)).ravel()
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_std, Y_std)
    return ridge.coef_

# ----------------------------
# Run mTRF
# ----------------------------
trf_pac = fit_mtrf(lf_signal_pac, sim_eeg_pac)
trf_aac = fit_mtrf(lf_envelope_aac, sim_eeg_aac)
trf_ppc = fit_mtrf(lf_phase_ppc, sim_eeg_ppc)

# ----------------------------
# Plot all TRFs
# ----------------------------
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.plot(lags * 1000, trf_pac, label='PAC mTRF')
plt.title('Phase-Amplitude Coupling')
plt.axhline(0, color='gray', linestyle='--')
plt.xlabel('Lag (ms)')
plt.ylabel('TRF weight')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(lags * 1000, trf_aac, label='AAC mTRF', color='darkorange')
plt.title('Amplitude-Amplitude Coupling')
plt.axhline(0, color='gray', linestyle='--')
plt.xlabel('Lag (ms)')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(lags * 1000, trf_ppc, label='PPC mTRF', color='teal')
plt.title('Phase-Phase Coupling')
plt.axhline(0, color='gray', linestyle='--')
plt.xlabel('Lag (ms)')
plt.legend()

plt.tight_layout()
plt.show()


# Statistics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.utils import shuffle
from sklearn.preprocessing import StandardScaler

# Simulation setup (same as before)
sfreq = 1000
duration = 10
n_samples = int(sfreq * duration)
times = np.arange(n_samples) / sfreq

# Simulated coupling data
lf_signal_pac = np.sin(2 * np.pi * 4 * times)
lf_phase_pac = np.angle(np.exp(1j * 2 * np.pi * 4 * times))
hf_carrier = np.sin(2 * np.pi * 80 * times)
hf_envelope = 1 + 0.8 * np.cos(lf_phase_pac)
sim_eeg_pac = hf_envelope * hf_carrier + 0.5 * np.random.randn(n_samples)

lf_envelope_aac = np.abs(lf_signal_pac)
hf_signal_aac = np.sin(2 * np.pi * 40 * times)
sim_eeg_aac = hf_signal_aac * (1 + 0.5 * lf_envelope_aac) + 0.1 * np.random.randn(n_samples)

lf_phase_ppc = np.angle(np.exp(1j * 2 * np.pi * 4 * times))
hf_phase_ppc = np.angle(np.exp(1j * 2 * np.pi * 40 * times))
combined_phase = np.cos(lf_phase_ppc - hf_phase_ppc)
sim_eeg_ppc = combined_phase + 0.1 * np.random.randn(n_samples)

# Organize predictors and responses
predictors = {
    'PAC': lf_signal_pac,
    'AAC': lf_envelope_aac,
    'PPC': lf_phase_ppc
}
responses = {
    'PAC': sim_eeg_pac,
    'AAC': sim_eeg_aac,
    'PPC': sim_eeg_ppc
}

# Parameters
lags = np.arange(-100, 401)  # in samples (ms if sfreq=1000)
n_permutations = 100
alpha = 1.0

# Helper: make lagged matrix
def make_lagged_matrix(signal, lags):
    pad = np.max(np.abs(lags))
    padded = np.pad(signal, (pad,), mode='constant')
    return np.stack([
        padded[pad + lag : pad + lag + len(signal)] for lag in lags
    ], axis=1)

# Main loop
for key in predictors:
    X_raw = predictors[key]
    y_raw = responses[key]

    # Preprocess
    X_lagged = make_lagged_matrix(X_raw, lags)
    X_std = StandardScaler().fit_transform(X_lagged)
    y_std = StandardScaler().fit_transform(y_raw.reshape(-1, 1)).ravel()

    # Fit real model
    model = Ridge(alpha=alpha)
    model.fit(X_std, y_std)
    trf_real = model.coef_

    # Permutation test
    null_trfs = []
    for _ in range(n_permutations):
        y_perm = shuffle(y_std, random_state=None)
        model.fit(X_std, y_perm)
        null_trfs.append(model.coef_)
    null_trfs = np.array(null_trfs)

    # --- p-value computation ---
    # Two-sided: count how many permuted coefficients are >= |real|
    pvals = np.mean(np.abs(null_trfs) >= np.abs(trf_real), axis=0)
    significant = pvals < 0.05

    # --- Plot TRF + null ---
    plt.figure(figsize=(10, 4))
    plt.plot(lags, trf_real, label='Real TRF', color='blue')
    plt.fill_between(lags,
                     np.percentile(null_trfs, 2.5, axis=0),
                     np.percentile(null_trfs, 97.5, axis=0),
                     color='gray', alpha=0.3, label='95% CI')
    plt.plot(lags[significant], trf_real[significant], 'ro', label='p < 0.05', markersize=3)
    plt.axhline(0, color='gray', linestyle='--')
    plt.title(f'{key}: TRF with Permutation Test (p < 0.05)')
    plt.xlabel('Lag (ms)')
    plt.ylabel('TRF weight')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # --- Plot SNR ---
    null_var = np.var(null_trfs, axis=0)
    snr = trf_real**2 / (null_var + 1e-10)

    plt.figure(figsize=(10, 3))
    plt.plot(lags, snr, label='SNR', color='green')
    plt.axhline(1, color='gray', linestyle='--')
    plt.title(f'{key}: SNR from Permutations')
    plt.xlabel('Lag (ms)')
    plt.ylabel('SNR')
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.utils import shuffle
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

# ----------------------------
# Simulation Setup
# ----------------------------
sfreq = 1000
duration = 10
n_samples = int(sfreq * duration)
times = np.arange(n_samples) / sfreq

# Generate signals
lf_signal = np.sin(2 * np.pi * 4 * times)
lf_phase = np.angle(np.exp(1j * 2 * np.pi * 4 * times))
lf_envelope = np.abs(lf_signal)
hf_carrier_pac = np.sin(2 * np.pi * 80 * times)
hf_signal_aac = np.sin(2 * np.pi * 40 * times)
hf_phase = np.angle(np.exp(1j * 2 * np.pi * 40 * times))

# PAC
hf_envelope_pac = 1 + 0.8 * np.cos(lf_phase)
sim_eeg_pac = hf_envelope_pac * hf_carrier_pac + 0.5 * np.random.randn(n_samples)

# AAC
sim_eeg_aac = hf_signal_aac * (1 + 0.5 * lf_envelope) + 0.1 * np.random.randn(n_samples)

# PPC
combined_phase = np.cos(lf_phase - hf_phase)
sim_eeg_ppc = combined_phase + 0.1 * np.random.randn(n_samples)

# ----------------------------
# Utility functions
# ----------------------------
def make_lagged_matrix(signal, lags):
    pad = np.max(np.abs(lags))
    padded = np.pad(signal, (pad,), mode='constant')
    return np.stack([
        padded[pad + lag : pad + lag + len(signal)] for lag in lags
    ], axis=1)

def compute_pvals(trf, null_trfs):
    """Compute uncorrected and FDR-corrected p-values"""
    n_perm, n_lags = null_trfs.shape
    p_vals = np.array([
        np.mean(np.abs(null_trfs[:, i]) >= np.abs(trf[i]))
        for i in range(n_lags)
    ])
    _, pvals_fdr, _, _ = multipletests(p_vals, alpha=0.05, method='fdr_bh')
    return p_vals, pvals_fdr

def plot_pvals(p_raw, p_fdr, lags, title):
    plt.figure(figsize=(10, 3))
    plt.plot(lags, p_raw, label='Uncorrected')
    plt.plot(lags, p_fdr, label='FDR-corrected')
    plt.axhline(0.05, color='red', linestyle='--', label='p = 0.05')
    plt.xlabel('Lag (ms)')
    plt.ylabel('p-value')
    plt.title(f'{title} – Permutation-based p-values')
    plt.legend()
    plt.tight_layout()
    plt.show()

# ----------------------------
# Main TRF & Permutation Loop
# ----------------------------
lags = np.arange(-100, 401)  # in ms
n_permutations = 100
alpha = 1.0

cfc_results = {
    'PAC': (lf_signal, sim_eeg_pac),
    'AAC': (lf_envelope, sim_eeg_aac),
    'PPC': (lf_phase, sim_eeg_ppc),
}

trfs = {}
nulls = {}
pvals_raw = {}
pvals_fdr = {}

for key, (predictor, response) in cfc_results.items():
    X = make_lagged_matrix(predictor, lags)
    X_std = StandardScaler().fit_transform(X)
    y_std = StandardScaler().fit_transform(response.reshape(-1, 1)).ravel()

    model = Ridge(alpha=alpha)
    model.fit(X_std, y_std)
    trf_real = model.coef_
    trfs[key] = trf_real

    # Null distribution
    null_trfs = []
    for _ in range(n_permutations):
        y_shuffled = shuffle(y_std, random_state=None)
        model.fit(X_std, y_shuffled)
        null_trfs.append(model.coef_)
    null_trfs = np.array(null_trfs)
    nulls[key] = null_trfs

    # Compute p-values
    p_raw, p_fdr = compute_pvals(trf_real, null_trfs)
    pvals_raw[key] = p_raw
    pvals_fdr[key] = p_fdr

    # Plot TRF with significance
    plt.figure(figsize=(10, 4))
    plt.plot(lags, trf_real, label='TRF', color='blue')
    plt.fill_between(lags,
                     np.percentile(null_trfs, 2.5, axis=0),
                     np.percentile(null_trfs, 97.5, axis=0),
                     color='gray', alpha=0.3, label='95% Null CI')
    plt.plot(lags[p_fdr < 0.05], trf_real[p_fdr < 0.05], 'ro', markersize=3, label='FDR sig (p<0.05)')
    plt.axhline(0, color='gray', linestyle='--')
    plt.title(f'{key} – TRF with Permutation Null & FDR')
    plt.xlabel('Lag (ms)')
    plt.ylabel('TRF weight')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Plot p-values
    plot_pvals(p_raw, p_fdr, lags, title=key)
